# PitchBook Saved List → CSV (Playwright, Colab)

Este notebook usa Playwright para abrir tu Saved List en PitchBook, extraer todas las filas y guardar un CSV con fecha.

**Flujo recomendado**
1. Ejecuta **Setup**.
2. Ejecuta **Login manual (solo primera vez)** para guardar el perfil.
3. Ejecuta **Scrape** para extraer datos.
4. Ejecuta **Preview + descarga** para ver/descargar el CSV.

Si la sesión expira, repite la celda de login manual.


## 1) Configuración
Ajusta las variables y los selectores.

**Cómo encontrar selectores:** abre la Saved List, haz clic derecho sobre la tabla y usa **Inspect**.
- Busca el contenedor de la tabla, filas y botón de paginación.
- Copia un selector estable (por ejemplo, un atributo `data-*` o un `aria-label`).


In [ ]:
# ==== Configuración general ====
SAVED_LIST_URL = "https://pitchbook.com/your-saved-list-url"

# Opción recomendada: Google Drive
BASE_DIR = "/content/drive/MyDrive/pitchbook_automation/"
PROFILE_DIR = BASE_DIR + "pb_profile"
EXPORT_DIR = BASE_DIR + "exports"

# Si NO usas Google Drive, puedes usar rutas locales (se pierden al reiniciar Colab):
# BASE_DIR = "/content/pitchbook_automation/"
# PROFILE_DIR = BASE_DIR + "pb_profile"
# EXPORT_DIR = BASE_DIR + "exports"

# ==== Selectores (ajusta según la página) ====
# Ejemplos (cámbialos por los reales de tu Saved List)
TABLE_SELECTOR = "table"
ROW_SELECTOR = "table tbody tr"
NEXT_BUTTON_SELECTOR = "button[aria-label='Next']"

# Selector opcional para detectar login expirado
LOGIN_BUTTON_SELECTOR = "a:has-text('Sign in')"

# Seguridad: máximo de páginas a recorrer
MAX_PAGES = 200

# Esperas y reintentos
TABLE_WAIT_MS = 15000
RETRY_ATTEMPTS = 3
RETRY_SLEEP_SEC = 3


## 2) Setup
Instala dependencias y Playwright Chromium.


In [ ]:
!pip -q install playwright pandas
!playwright install chromium

import os

os.makedirs(PROFILE_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print("PROFILE_DIR:", PROFILE_DIR)
print("EXPORT_DIR:", EXPORT_DIR)


## (Opcional) Montar Google Drive
Si quieres persistir el perfil y los CSV, monta Drive y usa las rutas configuradas arriba.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3) Login manual (solo primera vez)
Abre un navegador visible, inicia sesión manualmente y guarda el perfil en `PROFILE_DIR`.

Cuando termines, vuelve aquí y presiona Enter en el prompt.


In [ ]:
import time
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    context = p.chromium.launch_persistent_context(PROFILE_DIR, headless=False)
    page = context.new_page()
    page.goto('https://pitchbook.com', wait_until='domcontentloaded')
    page.goto(SAVED_LIST_URL, wait_until='domcontentloaded')
    print('Inicia sesión manualmente en el navegador abierto.')
    input('Cuando hayas terminado, presiona Enter para cerrar el navegador...')
    context.close()

print('Perfil guardado en:', PROFILE_DIR)


## 4) Scrape
Abre el perfil guardado en modo headless, extrae todas las filas y guarda un CSV.


In [ ]:
import csv
import time
from datetime import datetime
from playwright.sync_api import sync_playwright, TimeoutError

def wait_for_table(page):
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            page.wait_for_selector(TABLE_SELECTOR, timeout=TABLE_WAIT_MS)
            return True
        except TimeoutError:
            print(f'Intento {attempt}/{RETRY_ATTEMPTS}: tabla no lista, reintentando...')
            time.sleep(RETRY_SLEEP_SEC)
    return False

def login_expired(page):
    if page.url and 'login' in page.url.lower():
        return True
    try:
        return page.locator(LOGIN_BUTTON_SELECTOR).first.is_visible()
    except Exception:
        return False

def extract_rows(page):
    rows = []
    row_elements = page.locator(ROW_SELECTOR)
    count = row_elements.count()
    for i in range(count):
        row = row_elements.nth(i)
        cells = row.locator('td').all_inner_texts()
        if not cells:
            cells = [row.inner_text()]
        rows.append([cell.strip() for cell in cells])
    return rows

def is_next_enabled(page):
    next_button = page.locator(NEXT_BUTTON_SELECTOR)
    if next_button.count() == 0:
        return False
    if next_button.is_disabled():
        return False
    return True

all_rows = []
page_count = 0

with sync_playwright() as p:
    context = p.chromium.launch_persistent_context(PROFILE_DIR, headless=True)
    page = context.new_page()
    page.goto(SAVED_LIST_URL, wait_until='domcontentloaded')

    if login_expired(page):
        context.close()
        raise RuntimeError('Parece que la sesión expiró. Ejecuta la celda de login manual.')

    if not wait_for_table(page):
        context.close()
        raise RuntimeError('No se encontró la tabla. Revisa los selectores.')

    while True:
        page_count += 1
        print(f'Extrayendo página {page_count}...')
        rows = extract_rows(page)
        all_rows.extend(rows)
        if page_count >= MAX_PAGES:
            print('Máximo de páginas alcanzado, deteniendo.')
            break
        if not is_next_enabled(page):
            print('No hay más páginas.')
            break
        page.locator(NEXT_BUTTON_SELECTOR).click()
        time.sleep(2)
        if not wait_for_table(page):
            print('Tabla no cargó tras paginar, deteniendo.')
            break

    context.close()

date_str = datetime.utcnow().strftime('%Y-%m-%d')
csv_path = f"{EXPORT_DIR}/pitchbook_deals_{date_str}.csv"

max_cols = max((len(r) for r in all_rows), default=0)
headers = [f"col_{i+1}" for i in range(max_cols)]

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    if max_cols:
        writer.writerow(headers)
    writer.writerows(all_rows)

print(f'Filas extraídas: {len(all_rows)}')
print(f'CSV guardado en: {csv_path}')


## 5) Preview + descarga
Muestra una vista previa y ofrece descarga si no usas Drive.


In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
display(df.head())

# Si no estás usando Drive, puedes descargar el CSV:
# from google.colab import files
# files.download(csv_path)
